In [ ]:
import psutil
import sys
# !{sys.executable} --version
# !{sys.executable} -m pip install shap --upgrade 
import joblib
import time
from shutil import copy
import numpy as np
import pandas as pd
#import tensorflow as tf
import os
import matplotlib.pyplot as plt
from mpl_toolkits.mplot3d import Axes3D
from matplotlib import cm
from sklearn.metrics import mean_absolute_error, r2_score

from glob import glob
import psi4
from helper_CC_ML_spacial import HelperCCEnergy
# from helper_CC_ML import *

from glob import glob


In [ ]:
data_path, model_path = glob(os.path.join(os.path.expanduser("~"),"out/faster_sto_3g_*_20.pkl"))
# data_path, model_path = glob(os.path.join(os.path.expanduser("~"),"out/faster_cc_pVDZ*60.pkl"))

In [ ]:
with open(model_path,'rb') as m:
    model = joblib.load(m)

with open(data_path,'rb') as d:
    data = joblib.load(d)

In [ ]:
model

In [ ]:
X_train = data['X_train']
X_test = data['X_test']

y_train = data['y_train']
y_test = data['y_test']

In [ ]:
# out = model.pr(X_train,y_train)

# These are the machine learning features, these will be useful later

In [ ]:
# properties=['Evir1', 'Hvir1', 'Jvir1', 'Kvir1', 'Evir2', 'Hvir2', 'Jvir2', 'Kvir2', 'Eocc1', 'Jocc1', 'Kocc1', 'Hocc1','Eocc2', 'Jocc2', 'Kocc2', 'Hocc2', 'Jia1', 'Jia2', 'Kia1', 'Kia2','diag', 'orbdiff', 'doublecheck', 't2start', 't2mag', 't2sign', 'Jia1mag', 'Jia2mag','Kia1mag', 'Kia2mag','t2']
properties=['Evir1', 'Hvir1', 'Jvir1', 'Kvir1', 'Evir2', 'Hvir2', 'Jvir2', 'Kvir2', 'Eocc1', 'Jocc1', 'Kocc1', 'Hocc1','Eocc2', 'Jocc2', 'Kocc2', 'Hocc2', 'Jia1', 'Jia2', 'Kia1', 'Kia2','diag', 'orbdiff', 'doublecheck', 't2start', 't2mag', 't2sign', 'Jia1mag', 'Jia2mag','Kia1mag', 'Kia2mag']

# Use the basis sets for both Psi4 and PySCF

In [ ]:

basis_sets = ['STO-3G','cc-pVDZ','aug-cc-pVDZ']

# Load in the xyz coordinates of the system and then run Psi4 with the Psi4Numpy code we use for DDCC

# TO-DO Max:
- [ ] Create a loop/function that gets the following properties out of the data:
    - [ ] Final energies
    - [ ] Histories (iterations, etc.)
    - [ ] Feature set

In [ ]:

# with open('../check_amplitudes/diatomics/NN.xyz','r') as f:
#     text=f.read()
# qmol = psi4.qcdb.Molecule.from_string(text, dtype='xyz')
# mol = psi4.geometry(qmol.create_psi4_string_from_molecule()+ 'symmetry c1')                
# with open('../machine_learning/data/formaldehyde10.xyz','r') as f:
#     text=f.read()
# mol = psi4.geometry(text)                
with open('../machine_learning/data/water10.xyz','r') as f:
    text=f.read()
mol = psi4.geometry(text)                

psi4.core.clean()
psi4.core.be_quiet()

psi4.set_options({'basis': basis_sets[0],
                  'scf_type':     'pk',
                  'reference':    'rohf',
                  'mp2_type':     'conv',
                  'e_convergence': 1e-8,
                  'd_convergence': 1e-8})

rhf_e, scf_wfn = psi4.energy('scf', return_wfn=True)
scf_e, scf_wfn = psi4.energy('scf', return_wfn=True)

A=HelperCCEnergy(mol, rhf_e, scf_wfn,freeze_core=True)
MP2T2=A.t2start
# check=False
# if check==True:
#     # t1 to zeroes
#     A.t1 = np.random.rand(*A.t1.shape)
#     # ML t2 amplitudes here
#     A.t2 = np.random.rand(*A.t2.shape)

In [ ]:
best5=["doublecheck","t2start","t2mag","orbdiff","diag"]

In [ ]:
# ML initialization
y_pred = model.predict(np.vstack([getattr(A,i).flatten() for i in best5]).T)

A.t1 = np.zeros((A.t1.shape))
# ML t2 amplitudes here
A.t2 = y_pred.reshape(*A.t2.shape)
A.t2start = y_pred.reshape(*A.t2.shape)

mlE_0 = A.compute_energy(iterate=False)
mlE = A.compute_energy()

mlHistory = A.history

In [ ]:
plt.plot(A.t2.flatten(),A.t2.flatten(),'k:',label='True')
plt.scatter(A.t2.flatten(),y_pred.flatten(),label='ML')
plt.scatter(A.t2.flatten(),A.t2start.flatten(),label='MP2')
plt.ylabel("Predicted")
plt.xlabel("Calculated")
plt.legend()
plt.show()

In [ ]:
mean_absolute_error(A.t2.flatten(),y_pred.flatten()), r2_score(A.t2.flatten(),y_pred.flatten())

In [ ]:
# Normal initialization
A.t1 = np.zeros((A.t1.shape))
# ML t2 amplitudes here
A.t2 = MP2T2
MP2E = A.compute_energy(iterate=False)

# Final energy
CCSDE = A.compute_energy()

# Calculation history
normalHistory = A.history


In [ ]:
# check=True

# if check==True:
#     A.t1 = np.random.rand(*A.t1.shape)
#     A.t2 = np.random.rand(*A.t2.shape)

# A.compute_energy()

# rdmHistory = A.history


In [ ]:
# Add the initial DDCC energy to the list
mlHistory.insert(0, (0,mlE_0,0,0))
# Add the MP2 energy to the list
normalHistory.insert(0, (0,MP2E,0,0))

In [ ]:
print("Devations from the converged CCSD energies in mEh")
print(f"MP2: {abs(MP2E-CCSDE) * 1e3:.4e} mEh")
print(f"DDCC before iterations: {abs(mlE_0-CCSDE) * 1e3:.4e} mEh")
print(f"DDCC after iterations: {abs(mlE - CCSDE) * 1e3:.4e} mEh")

In [ ]:
plt.plot(np.array(mlHistory)[:,0].T,np.array(mlHistory)[:,1].T-CCSDE,label='DDCCSD')
plt.plot(np.array(normalHistory)[:,0].T,np.array(normalHistory)[:,1].T-CCSDE,label='CCSD')
plt.hlines(MP2E-CCSDE,-1,50,color='r',label='MP2')
plt.xlim(0-0.1,len(normalHistory)-0.9)
plt.ylabel("Energy Deviation (E$_{h}$)")
plt.xlabel("Iterations")
plt.legend()
plt.title('Convergence vs. Deviations from CCSD')
plt.tight_layout()
plt.show()